# పాఠం 10 - ఉత్పత్తిలో AI ఏజెంట్లు

ఈ పాఠంలో మీరు Microsoft Agent Framework (Python) ఉపయోగించి AI ఏజెంట్ల కోసం **ఉత్పత్తి నమూనాలు** నేర్చుకుంటారు. మేము కవర్ చేస్తాం:

- **పర్యవేక్షణ** — ఏజెంట్ పరస్పర చర్యలకు సమయం మరియు లాగింగ్ జోడించడం
- **మూల్యాంకనం** — ప్రతిస్పందన నాణ్యతను అంచనా వేసేందుకు ఒక మూల్యాంకన ఏజెంట్ ఉపయోగించడం
- **ఖర్చు నిర్వహణ** — టోకెన్ ఆప్టిమైజేషన్ మరియు మోడల్ ఎంపిక కోసం వ్యూహాలు

ఈ సన్నివేశం ఒక **ప్రయాణ ఏజెంట్** గురించి, ఇది వినియోగదారులకి ప్రయాణాలు ప్లాన్ చేయడంలో సహాయపడుతుంది, పర్యవేక్షణ మరియు మూల్యాంకనం పైకి జతచేయబడింది.


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ఉత్పత్తి ఆలోచనలు

నమూనాలనుండి AI ఏజెంట్లను ఉత్పత్తికి తరలించడం మూడు మూలస్తంభాలకు శ్రద్ధ అవసరం:

1. **పరిశీలన** — ఏజెంట్ ఏమి చేస్తున్నదో, ఎంతసేపు తీసుకుంటుందో, ఎలాంటి టూల్స్ పిలుస్తుందో చూడగలిగే వీలుండాలి. ట్రేసింగ్ మరియు లాగింగ్ లేకపోతే ఉత్పత్తి సమస్యలను డీబగ్ చేయడంలాగే అసాధ్యం.

2. **అంచనా** — స్వయంచాలక నాణ్యత తనిఖీలు ఏజెంట్ యొక్క ప్రత్యుత్తరాలు సమయానుకూలంగా సరిగ్గా, పూర్తి గా మరియు సహాయకంగా ఉండటాన్ని నిర్ధారిస్తాయి. అంచనా ఏజెంట్ ప్రతిస్పందనా ప్రమాణాలపై స్కోరు చేయవచ్చు.

3. **ఖర్చు నిర్వహణ** — టోకెన్ వినియోగం నేరుగా ఖర్చును ప్రభావితం చేస్తుంది. ప్రంప్ట్ ఆప్టిమైజేషన్, మోడల్ ఎంపిక, మరియు క్యాచింగ్ వంటి వ్యూహాలు నాణ్యతను త్యాగం చేయకుండా ఖర్చులను నియంత్రించడంలో సహాయపడతాయి.


## ఒక ఆబ్జర్వబుల్ ఏజెంట్‌ను నిర్మించడం

మేము ప్రయాణ సాధనాలను నిర్వచించి, ఆలస్యం ను పర్యవేక్షించేందుకు ఏజెంట్ కాల్‌ను సమయం తో సర్దుబాటు చేస్తాము. ఉత్పత్తి సందర్భంలో మీరు OpenTelemetry లేదా ఇలాంటి ట్రేసింగ్ బ్యాక్‌ఎండ్‌తో అనుసంధానం చేస్తారు.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## మూల్యాంకన నమూనాలు

సాధారణ ఉత్పత్తి నమూనా అనేది రెండవ ఏజెంట్‌ను **మూల్యాంకకుడిగా** ఉపయోగించడం. మూల్యాంకకుడు ప్రాథమిక ఏజెంట్ యొక్క ప్రతిస్పందనను పూర్తి స్థితి, ఖచ్చితత్వం మరియు సహాయప్రాయత వంటి ముందుగా నిర్వచించిన ప్రమాణాలను ఆధారంగా స్కోరు చేస్తాడు.

ఇది సాధ్యమవుతుంది:
- వినియోగదారుల వరకు ప్రతిస్పందనలు చేరముందు ఆటోమేటెడ్ క్వాలిటీ గేట్లు
- ప్రాంప్ట్‌లు లేదా మోడల్స్ మారినపుడు పరావర్తీ గుర్తింపు
- ఏజెంట్ పనితీరు సమయం గడిచేకుండా నిరంతరమైన పర్యవేక్షణ


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ఖర్చు నిర్వహణ వ్యూహాలు

ఉత్పత్తి AI ఏజెంట్ల కోసం ఖర్చులు నియంత్రించడం అత్యంత ముఖ్యం. ఇక్కడ ముఖ్యమైన వ్యూహాలు ఉన్నాయి:

| వ్యూహం | వివరణ |
|---|---|
| **ప్రాంప్ట్ ఆప్టిమైజేషన్** | సిస్టమ్ సూచనలను సంక్షిప్తంగా ఉంచండి. ఇన్పుట్ టోకెన్లను తగ్గించడానికి అప్రయోజనకరమైన సందర్భాన్ని తొలగించండి. |
    "| **మోడల్ ఎంపిక** | క్లాసిఫికేషన్ లేదా ఎక్స్‌ట్రాక్షన్ లాంటి సులభ పనుల కోసం చిన్న, తగ్గిన ఖర్చుతో మోడళ్లు (ఉదాహరణకు GPT-4.1-mini) ఉపయోగించండి, క్లిష్టమైన తర్కం కోసం పెద్ద మోడళ్లను రిజర్వు చేయండి. |\n",
| **క్యాచింగ్** | పరికరం ఫలితాలు మరియు తరచూ అడిగే ప్రశ్నలను క్యాచ్ చేసి మళ్లీ API కాల్స్ చేయకుండా ఉండండి. |
| **టోకెన్ బడ్జెట్లు** | అనుకోని పొడవైన ప్రతిస్పందనలను నివారించడానికి `max_tokens` పరిమితులను సెట్ చేయండి. |
| **బ్యాచింగ్** | సాధ్యమైనప్పుడు ఒకే API కాల్‌లో బహుళ వినియోగదారు ప్రశ్నలను సమూహీకరించండి. |

ప్రాక్టీస్‌లో, ఒక స్థాయినిర్ధారణ విధానం బాగా పనిచేస్తుంది: సరళమైన అభ్యర్ధనలు వేగంగా, తక్కువ ఖర్చుతో కూడిన మోడల్‌కి రూట్ చేయండి మరియు కేవలం క్లిష్ట ప్రశ్నలకే ఎక్కువ సామర్థ్యం గల మోడల్‌ను ఆక్రమించండి.


## సంగ్రహం

ఈ పాఠంలో మీరు ఎలా నేర్చుకున్నారంటే:

1. **ఎజెంట్ పరస్పర చర్యలకు ట్రేసింగ్ మరియు మానిటరింగ్ కోసం గడువు మరియు లాగింగ్‌తో** ఆబ్జర్వబిలిటీని జోడించడం.
2. **పూర్ణత, ఖచ్చితత్వం మరియు సహాయదాయకత స్కోర్ చేసే సంవేదక ఎజెంట్ ఉపయోగించి** ఎజెంట్ ప్రతిస్పందనలను స్వయంచాలకంగా అంచనా వేయడం.
3. **ప్రాంప్ట్ ఆప్టిమైజేషన్, మోడల్ ఎంపిక, కాచింగ్ మరియు టోకెన్ బడ్జెట్‌ల ద్వారా** ఖర్చులను నిర్వహించడం.

ఈ ప్రొడక్షన్ నమూనాలు మీ AI ఎజెంట్లు విశ్వసనీయమైనవి, కొలవదగినవి మరియు పెద్ద మొత్తంలో ధరల చర్యలో ఉన్నాయని నిర్ధారించడంలో సహాయపడతాయి.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
